# Import

In [243]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [244]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Dependency
* DGIDB_hypergraph
* DDBC

# Prep

## Loading variables

In [245]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [246]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [247]:
# Loading result graph and communities
with open(f"{DISEASE_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{DISEASE_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{DISEASE_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [248]:
for c in communities_selected:
    print(len(c))

1025
983
1201
1035
1111
748
666
534
485
342
300


## Helpful functions (big object, drop NAN)

In [249]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [250]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [251]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['55858', '10522', '55884', '55186', '84272', '57222', '10953', '5635', '10314', '81627', '9019', '50999', '23029', '6666', '7095', '7701', '113419', '23484', '54978', '55608', '81572', '140890', '57409', '79073', '27230', '157567', '10180', '9325', '9584', '51290', '9689', '64756', '23215', '222658', '5813', '9991', '9747', '9213', '8073', '10807', '11333', '51016', '4076', '51596', '55556', '51430', '54765', '64327', '9980', '29896', '51326', '26268', '9240', '9993', '23392', '64746', '64118', '57621', '54542', '262', '57720', '51397', '55005', '11163', '10181', '136319', '91404', '10330', '104472715', '10116', '9898', '58497', '7993', '9202', '25843', '51088', '51663', '9643', '56061', '9522', '23200', '23174', '7511', '23648', '387263', '8545', '147007', '7536', '10771', '55325', '4820', '23065', '10311', '7267', '9779', '6767', '6651', '399979', '124565', '157922', '58517', '23151', '30000', '7873', '10300', '57700', '23276', '10428', '81688', '84939', '55251', '7163', '64418', '

In [252]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['3', '47', '48', '50', '52', '56', '100', '132', '159', '162', '164', '204', '262', '267', '271', '286', '291', '292', '293', '310', '353', '372', '377', '378', '381', '400', '402', '403', '439', '473', '498', '506', '509', '513', '514', '515', '516', '517', '518', '520', '521', '522', '539', '547', '549', '550', '582', '583', '585', '586', '587', '617', '631', '662', '734', '738', '740', '745', '755', '811', '819', '821', '829', '830', '832', '833', '889', '955', '989', '1039', '1054', '1068', '1080', '1120', '1122', '1155', '1174', '1176', '1182', '1192', '1198', '1201', '1209', '1314', '1315', '1327', '1329', '1337', '1339', '1340', '1345', '1346', '1347', '1349', '1350', '1351', '1352', '1353', '1355', '1371', '1406', '1411', '1427', '1429', '1431', '1468', '1527', '1537', '1539', '1603', '1629', '1639', '1650', '1654', '1656', '1657', '1716', '1723', '1725', '1731', '1737', '1738', '1743', '1797', '1801', '1819', '1877', '1891', '1939', '1951', '1979', '1983', '1984', '2023', '2

## NCBI to HGNC

In [253]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [254]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [255]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [256]:
print(len(COMMUNITIES_HGNC))

11


In [257]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 3 NaN entries
Community 2: dropped 1 NaN entries
Community 3: dropped 1 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 1 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries

Total dropped across all communities: 7
Community 0: dropped 5 NaN entries
Community 1: dropped 8 NaN entries
Community 2: dropped 9 NaN entries
Community 3: dropped 5 NaN entries
Community 4: dropped 1 NaN entries
Community 5: dropped 3 NaN entries
Community 6: dropped 2 NaN entries
Community 7: dropped 3 NaN entries
Community 8: dropped 2 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries

Total dropped across all communities: 39


In [258]:
with open(f"{DISEASE_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{DISEASE_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [259]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

11
15


# Categoization Prep

### GO-slim

In [260]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [261]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [262]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [263]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [264]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [265]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [266]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [267]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [268]:
TERM_SCORE_CAP = 1e-5
PERCENTAGE = 0.1

In [269]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Adjusted P-value'], ascending=True)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [270]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [271]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [272]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 1025
Number of filtered terms: 13
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_105056\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
2029,0,Ubiquitin-Protein Transferase Activity (GO:0004842),60/412,4.984610e-11,[catalytic activity]
0,0,Golgi Vesicle Transport (GO:0048193),36/197,4.935722e-08,"[localization, cellular process]"
1,0,Protein Ubiquitination (GO:0016567),56/434,1.809751e-07,[cellular process]
2,0,Ubiquitin-Dependent Protein Catabolic Process (GO:0006511),50/367,1.809751e-07,[cellular process]
3,0,Protein Transport (GO:0015031),45/313,1.858556e-07,[localization]
4,0,Protein Localization (GO:0008104),48/351,2.181298e-07,[localization]
2457,0,cullin-RING Ubiquitin Ligase Complex (GO:0031461),30/174,1.122938e-06,[protein-containing complex]
5,0,Proteasome-Mediated Ubiquitin-Dependent Protein Catabolic Process (GO:0043161),43/319,2.243102e-06,[cellular process]
6,0,Intracellular Protein Transport (GO:0006886),43/325,3.066751e-06,"[localization, cellular process]"
7,0,post-Golgi Vesicle-Mediated Transport (GO:0006892),16/56,3.066751e-06,"[localization, cellular process]"


Size of community: 1200
Number of filtered terms: 130
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Cytokine-Mediated Signaling Pathway (GO:0019221),95/257,1.113217e-46,"[cellular process, biological regulation]"
1,2,Cellular Response To Cytokine Stimulus (GO:0071345),94/308,2.855925e-38,[response to stimulus]
2,2,Inflammatory Response (GO:0006954),80/236,3.967115e-36,[response to stimulus]
2741,2,Chemokine Receptor Binding (GO:0042379),37/50,2.624805e-32,[binding]
3,2,Neutrophil Chemotaxis (GO:0030593),43/70,3.631976e-32,"[cellular process, locomotion, immune system process]"
2742,2,Chemokine Activity (GO:0008009),35/46,1.630087e-31,"[binding, molecular function regulator activity]"
4,2,Granulocyte Chemotaxis (GO:0071621),43/73,3.419062e-31,"[cellular process, locomotion, immune system process]"
5,2,Neutrophil Migration (GO:1990266),44/77,3.419062e-31,"[cellular process, immune system process]"
6,2,Response To Type II Interferon (GO:0034341),44/80,2.831243e-30,[response to stimulus]
2743,2,Cytokine Activity (GO:0005125),63/178,4.426208e-30,"[binding, molecular function regulator activity]"


...

Size of community: 484
Number of filtered terms: 74
Number of unmapped terms: 4


,Community Index,Term,Overlap,Adjusted P-value,Category
1246,8,Peroxisomal Matrix (GO:0005782),29/49,5.395830e-33,[cellular anatomical structure]
1245,8,Microbody Lumen (GO:0031907),29/49,5.395830e-33,[cellular anatomical structure]
0,8,Steroid Metabolic Process (GO:0008202),36/92,2.369092e-31,[cellular process]
1247,8,Peroxisome (GO:0005777),38/129,2.877702e-29,[cellular anatomical structure]
976,8,"Oxidoreductase Activity, Acting On The CH-OH Group Of Donors, NAD Or NADP As Acceptor (GO:0016616)",34/95,1.685256e-28,[catalytic activity]
1,8,Fatty Acid Metabolic Process (GO:0006631),37/122,7.603102e-28,[cellular process]
2,8,Fatty Acid Beta-Oxidation (GO:0006635),23/49,4.244016e-22,[cellular process]
3,8,Monocarboxylic Acid Metabolic Process (GO:0032787),28/97,2.473644e-20,[cellular process]
4,8,Fatty Acid Oxidation (GO:0019395),21/52,1.398574e-18,[cellular process]
5,8,Long-Chain Fatty Acid Metabolic Process (GO:0001676),23/78,6.563095e-17,[cellular process]


Size of community: 300
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
21,10,Olfactory Receptor Activity (GO:0004984),250/362,0.000000e+00,[molecular transducer activity]
0,10,Sensory Perception Of Smell (GO:0007608),149/230,3.020874e-227,[multicellular organismal process]
1,10,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),90/141,1.383285e-131,[response to stimulus]
2,10,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),89/139,2.041730e-130,[response to stimulus]
3,10,Sensory Perception Of Chemical Stimulus (GO:0007606),67/110,5.372131e-95,[multicellular organismal process]


9 out of 11 communities had significant GO terms.


In [273]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1025,Ubiquitin-Protein Transferase Activity (GO:000...,60/412,4.984610e-11,[catalytic activity],GO_Molecular_Function_2023,2.323827e-13,0.0,0.0,3.289508,9.569307e+01,RNF10;UBE2D4;RNF13;PPP1R11;RNF14;LTN1;UBE3B;FB...,0.145631
1,0,1025,Golgi Vesicle Transport (GO:0048193),36/197,4.935722e-08,"[localization, cellular process]",GO_Biological_Process_2023,2.433788e-11,0.0,0.0,4.253647,1.039548e+02,ARF3;ARF4;NBAS;TMED10;SAR1A;ATL3;PITPNB;PDCD6;...,0.182741
2,0,1025,Protein Ubiquitination (GO:0016567),56/434,1.809751e-07,[cellular process],GO_Biological_Process_2023,1.874903e-10,0.0,0.0,2.843252,6.368115e+01,RNF10;UBE2D4;RNF13;RNF14;FBXO28;UBE3B;DDA1;RPG...,0.129032
3,0,1025,Ubiquitin-Dependent Protein Catabolic Process ...,50/367,1.809751e-07,[cellular process],GO_Biological_Process_2023,2.677146e-10,0.0,0.0,3.018361,6.652800e+01,PPP1R11;UBE2D4;RNF13;RNF14;UBE3B;FBXO21;RPGR;H...,0.136240
4,0,1025,Protein Transport (GO:0015031),45/313,1.858556e-07,[localization],GO_Biological_Process_2023,3.665792e-10,0.0,0.0,3.205205,6.963887e+01,ARF3;ARF4;STX12;C17ORF75;TMED10;SAR1A;STX16;PD...,0.143770
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,10,300,Olfactory Receptor Activity (GO:0004984),250/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,874.464286,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR2M5;OR52N1;OR2M4;OR2...,0.690608
876,10,300,Sensory Perception Of Smell (GO:0007608),149/230,3.020874e-227,[multicellular organismal process],GO_Biological_Process_2023,1.438511e-228,0.0,0.0,239.001799,1.253865e+05,OR8I2;OR1C1;OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T...,0.647826
877,10,300,Detection Of Chemical Stimulus Involved In Sen...,90/141,1.383285e-131,[response to stimulus],GO_Biological_Process_2023,1.317414e-132,0.0,0.0,165.117647,5.014054e+04,OR10J1;OR2A1;OR10J3;OR2M7;OR13H1;OR2M5;OR2M4;O...,0.638298
878,10,300,Detection Of Chemical Stimulus Involved In Sen...,89/139,2.041730e-130,[response to stimulus],GO_Biological_Process_2023,2.916757e-131,0.0,0.0,165.767773,4.982452e+04,OR10J1;OR2A1;OR10J3;OR2M7;OR13H1;OR2M5;OR2M4;O...,0.640288


In [274]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1025,"[AKIRIN2, AP1G2, AP3D1, AP4M1, AP4S1, APPBP2, ...",197,828
1,1,980,[],0,980
2,2,1200,"[ACE2, ACKR4, ACOD1, ADAM8, ADAR, ADGRG3, AHR,...",502,698
3,3,1034,"[AARS1, AARSD1, AATF, ABCE1, ABT1, ACTR6, ADNP...",754,280
4,4,1111,"[AAK1, ABHD17A, ABHD17B, ABHD17C, ABHD6, ABI1,...",951,160
5,5,748,"[A2M, ABI3BP, ADAM12, ADAM19, ADAM23, ADAM9, A...",315,433
6,6,666,"[ABRAXAS2, ACTL6A, ACTL6B, AEBP2, AGO2, AIFM2,...",533,133
7,7,534,"[ADGRL3, ADORA1, ADORA2A, ADRA2A, ADRA2C, ADRB...",288,246
8,8,484,"[AASS, ABAT, ABCA6, ABCA8, ABCB4, ABCC2, ABCC3...",220,264
9,9,341,[],0,341


### KEGG

In [275]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [276]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1200
Number of filtered terms: 18
Number of unmapped terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_105056\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Cytokine-cytokine receptor interaction,144/295,3.422747e-94,[Signaling molecules and interaction]
1,2,Viral protein interaction with cytokine and cytokine receptor,65/100,1.008608e-52,[Signaling molecules and interaction]
2,2,NF-kappa B signaling pathway,35/104,7.183563e-16,[Signal transduction]
3,2,Hematopoietic cell lineage,33/99,6.382783e-15,[Immune system]
4,2,Chemokine signaling pathway,45/192,5.933953e-14,[Immune system]
5,2,Natural killer cell mediated cytotoxicity,35/131,1.054313e-12,[Immune system]
6,2,Rheumatoid arthritis,29/93,1.829496e-12,[Immune disease]
7,2,Antigen processing and presentation,26/78,5.400052e-12,[Immune system]
8,2,IL-17 signaling pathway,27/94,9.253321e-11,[Immune system]
9,2,JAK-STAT signaling pathway,36/162,1.079481e-10,[Signal transduction]


Size of community: 1034
Number of filtered terms: 5
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Category
0,3,Spliceosome,73/150,1.777765e-51,[Transcription]
1,3,Ribosome biogenesis in eukaryotes,42/108,1.141640e-24,[Translation]
2,3,RNA transport,38/186,6.821590e-12,[]
3,3,Cell cycle,30/124,1.736156e-11,[Cell growth and death]
4,3,mRNA surveillance pathway,25/98,3.191669e-10,[Translation]


...

Size of community: 484
Number of filtered terms: 26
Number of unmapped terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
0,8,Metabolism of xenobiotics by cytochrome P450,34/76,6.794524e-33,[Xenobiotics biodegradation and metabolism]
1,8,Drug metabolism,38/108,2.182382e-32,[]
2,8,Peroxisome,32/82,4.837713e-29,[Transport and catabolism]
3,8,Retinol metabolism,28/68,2.825415e-26,[Metabolism of cofactors and vitamins]
4,8,Fatty acid degradation,22/43,2.456888e-23,[Lipid metabolism]
5,8,Pyruvate metabolism,19/47,9.274391e-18,[Carbohydrate metabolism]
6,8,Tyrosine metabolism,15/36,2.484902e-14,[Amino acid metabolism]
7,8,Steroid hormone biosynthesis,18/61,4.018649e-14,[Lipid metabolism]
8,8,PPAR signaling pathway,19/74,1.039110e-13,[Endocrine system]
9,8,Biosynthesis of unsaturated fatty acids,13/27,1.323537e-13,[Lipid metabolism]


Size of community: 300
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Olfactory transduction,298/440,0.0,[Sensory system]


8 out of 11 communities had significant GO terms.


In [277]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,2,1200,Cytokine-cytokine receptor interaction,144/295,3.422747e-94,[Signaling molecules and interaction],KEGG_2021_Human,1.764302e-96,0.0,0.0,16.841361,3713.190155,CNTFR;IL1RN;CSF3;TNFRSF6B;CSF3R;CSF1;CXCL17;IL...,0.488136
1,2,1200,Viral protein interaction with cytokine and cy...,65/100,1.008608e-52,[Signaling molecules and interaction],KEGG_2021_Human,1.039802e-54,0.0,0.0,30.704216,3816.551434,CXCL6;CXCL9;CXCL8;IL20;CSF1;IL24;CXCL1;CXCL13;...,0.650000
2,2,1200,NF-kappa B signaling pathway,35/104,7.183563e-16,[Signal transduction],KEGG_2021_Human,1.110860e-17,0.0,0.0,8.155564,318.383523,CCL13;CD40;CXCL8;EDA;BCL2A1;TNFAIP3;CXCL1;TNFR...,0.336538
3,2,1200,Hematopoietic cell lineage,33/99,6.382783e-15,[Immune system],KEGG_2021_Human,1.316038e-16,0.0,0.0,8.026564,293.505240,CSF3;GYPA;CSF3R;ITGAM;CSF1;CD1E;CD1D;CD1C;CSF2...,0.333333
4,2,1200,Chemokine signaling pathway,45/192,5.933953e-14,[Immune system],KEGG_2021_Human,1.529369e-15,0.0,0.0,4.943811,168.652788,CCL14;CX3CR1;CXCL6;CCL13;CXCL9;CXCL8;CCL11;CXC...,0.234375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,8,484,ABC transporters,10/45,4.400686e-07,[Membrane transport],KEGG_2021_Human,9.371831e-08,0.0,0.0,11.742616,190.030430,ABCC3;ABCD2;ABCB1;ABCC2;ABCA6;ABCB4;ABCC6;ABCA...,0.222222
162,8,484,Fatty acid elongation,8/27,7.400541e-07,[Lipid metabolism],KEGG_2021_Human,1.644565e-07,0.0,0.0,17.246351,269.398696,ACAA2;ELOVL2;ELOVL3;THEM5;ACOT2;ACOT1;HSD17B12...,0.296296
163,8,484,Propanoate metabolism,8/34,5.006275e-06,[Carbohydrate metabolism],KEGG_2021_Human,1.158860e-06,0.0,0.0,12.598578,172.198292,MCEE;ALDH6A1;ACOX1;EHHADH;ABAT;MLYCD;ACOX3;ACSS1,0.235294
164,8,484,Phenylalanine metabolism,6/17,7.979730e-06,[Amino acid metabolism],KEGG_2021_Human,1.921046e-06,0.0,0.0,22.257512,292.967638,ALDH3B2;MAOB;MAOA;ALDH3B1;PAH;GLYAT,0.352941


In [278]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1025,[],0,1025
1,1,980,[],0,980
2,2,1200,"[ACKR4, AIM2, ANPEP, ANTXR2, B2M, BCL2A1, BCL3...",295,905
3,3,1034,"[BCAS2, BMS1, BUB1, BUB1B, BUB3, BUD31, CASC3,...",178,856
4,4,1111,"[ABHD6, ABI1, ABI2, ABL2, ABLIM2, ABLIM3, ACAP...",730,381
5,5,748,"[AGRN, COL11A1, COL12A1, COL13A1, COL14A1, COL...",36,712
6,6,666,"[ACTL6A, ACTL6B, ARID1A, ARID1B, ARID2, ATXN2L...",239,427
7,7,534,"[ADORA1, ADORA2A, ADRA1A, ADRA2A, ADRA2C, ADRB...",100,434
8,8,484,"[ABAT, ABCA6, ABCA8, ABCB1, ABCB4, ABCC2, ABCC...",180,304
9,9,341,[],0,341


### Reactome

In [279]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [280]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 1025
Number of filtered terms: 5
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_105056\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,Antigen Processing: Ubiquitination And Proteasome Degradation R-HSA-983168,55/307,1.128290e-13,[Immune System]
1,0,Class I MHC Mediated Antigen Processing And Presentation R-HSA-983169,55/378,4.387452e-10,[Immune System]
2,0,Neddylation R-HSA-8951664,41/237,7.566094e-10,[Metabolism of proteins]
3,0,Intra-Golgi And Retrograde Golgi-to-ER Traffic R-HSA-6811442,34/181,3.558480e-09,[Vesicle-mediated transport]
4,0,Retrograde Transport At Trans-Golgi-Network R-HSA-6811440,14/48,5.312353e-06,[Vesicle-mediated transport]


Size of community: 1200
Number of filtered terms: 27
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Immune System R-HSA-168256,371/1943,1.767154e-98,[Immune System]
1,2,Cytokine Signaling In Immune System R-HSA-1280215,204/702,5.276037e-84,[Immune System]
2,2,Immunoregulatory Interactions Between A Lymphoid And A non-Lymphoid Cell R-HSA-198933,63/123,1.017779e-41,[Immune System]
3,2,Signaling By Interleukins R-HSA-449147,115/453,1.657291e-39,[Immune System]
4,2,Chemokine Receptors Bind Chemokines R-HSA-380108,40/56,1.417278e-34,[Signal Transduction]
5,2,TNFs Bind Their Physiological Receptors R-HSA-5669034,26/29,4.061332e-27,[Immune System]
6,2,Interferon Alpha/Beta Signaling R-HSA-909733,37/72,1.641324e-24,[Immune System]
7,2,Neutrophil Degranulation R-HSA-6798695,95/468,1.641324e-24,[Immune System]
8,2,Interferon Signaling R-HSA-913531,60/200,1.667438e-24,[Immune System]
9,2,Interleukin-10 Signaling R-HSA-6783783,29/45,4.092809e-23,[Immune System]


...

Size of community: 484
Number of filtered terms: 41
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,8,Metabolism R-HSA-1430728,225/2049,4.172829e-93,[Metabolism]
1,8,Biological Oxidations R-HSA-211859,83/218,1.894052e-75,[Metabolism]
2,8,Metabolism Of Lipids R-HSA-556833,114/732,3.894460e-58,[Metabolism]
3,8,Phase I - Functionalization Of Compounds R-HSA-211945,53/104,4.250396e-56,[Metabolism]
4,8,Fatty Acid Metabolism R-HSA-8978868,63/173,1.514265e-55,[Metabolism]
5,8,Peroxisomal Lipid Metabolism R-HSA-390918,21/29,1.200198e-26,[Metabolism]
6,8,Peroxisomal Protein Import R-HSA-9033241,27/63,9.428256e-26,[Protein localization]
7,8,Cytochrome P450 - Arranged By Substrate Type R-HSA-211897,27/65,2.334432e-25,[Metabolism]
8,8,Metabolism Of Steroids R-HSA-8957322,37/153,3.959546e-25,[Metabolism]
9,8,Protein Localization R-HSA-9609507,34/164,9.108702e-21,[Protein localization]


Size of community: 300
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Sensory Perception R-HSA-9709957,293/616,0.0,[Sensory Perception]
1,10,Olfactory Signaling Pathway R-HSA-381753,293/401,0.0,[Sensory Perception]
2,10,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,293/393,0.0,[Sensory Perception]


9 out of 11 communities had significant GO terms.


In [281]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1025,Antigen Processing: Ubiquitination And Proteas...,55/307,1.128290e-13,[Immune System],Reactome_2022,3.151649e-16,0.0,0.0,4.212752,150.367579,UBE2D4;RNF14;LTN1;UBE3B;FBXO21;HERC4;NPEPPS;RN...,0.179153
1,0,1025,Class I MHC Mediated Antigen Processing And Pr...,55/378,4.387452e-10,[Immune System],Reactome_2022,2.451091e-12,0.0,0.0,3.274265,87.535801,UBE2D4;RNF14;LTN1;UBE3B;FBXO21;HERC4;NPEPPS;RN...,0.145503
2,0,1025,Neddylation R-HSA-8951664,41/237,7.566094e-10,[Metabolism of proteins],Reactome_2022,6.340303e-12,0.0,0.0,3.992134,102.933570,DCUN1D3;DCUN1D4;DCUN1D1;ASB13;TULP4;DCAF5;FBXO...,0.172996
3,0,1025,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,34/181,3.558480e-09,[Vesicle-mediated transport],Reactome_2022,3.975956e-11,0.0,0.0,4.394324,105.236032,ARF3;ARF4;SCOC;NBAS;TMF1;STX16;GOSR2;GOSR1;GOL...,0.187845
4,0,1025,Retrograde Transport At Trans-Golgi-Network R-...,14/48,5.312353e-06,[Vesicle-mediated transport],Reactome_2022,7.419488e-08,0.0,0.0,7.714377,126.643618,SCOC;NAA30;TMF1;STX16;M6PR;RABEPK;TGOLN2;ARFRP...,0.291667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
621,8,484,Synthesis Of Bile Acids And Bile Salts Via 7Al...,7/24,7.395871e-06,[Metabolism],Reactome_2022,1.127252e-06,0.0,0.0,16.832285,230.530397,CYP27A1;AMACR;ACOX2;AKR1C1;BAAT;CYP7B1;SLC27A2,0.291667
622,8,484,Heme Degradation R-HSA-189483,6/16,8.127142e-06,[Metabolism],Reactome_2022,1.268922e-06,0.0,0.0,24.484519,332.434710,FABP1;ABCC2;SLCO2B1;ALB;GSTA1;SLCO1B3,0.375000
623,10,300,Sensory Perception R-HSA-9709957,293/616,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2511.039805,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR11H4;OR2M5;OR52N1;OR...,0.475649
624,10,300,Olfactory Signaling Pathway R-HSA-381753,293/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,7593.195767,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR11H4;OR2M5;OR52N1;OR...,0.730673


In [282]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1025,"[ARF3, ARF4, ARFRP1, ARIH2, ASB1, ASB13, ASB6,...",106,919
1,1,980,[],0,980
2,2,1200,"[ACKR4, ADA2, ADAM8, ADAR, ADGRE1, ADGRE2, ADG...",419,781
3,3,1034,"[AARS1, AHCTF1, AIMP2, APEH, ARPP19, AURKA, AU...",412,622
4,4,1111,"[AAMP, ABHD17A, ABHD17B, ABHD17C, ABHD6, ABI1,...",869,242
5,5,748,"[ADAM12, ADAM19, ADAM9, ADAMTS1, ADAMTS2, ADAM...",141,607
6,6,666,"[ABRAXAS2, ACTL6A, ACTL6B, AEBP2, AGO2, AIFM2,...",536,130
7,7,534,"[ADORA1, ADORA2A, ADRA1A, ADRA2A, ADRA2C, ADRB...",129,405
8,8,484,"[AADAC, AASS, ABCB1, ABCB4, ABCC2, ABCC3, ABCD...",233,251
9,9,341,[],0,341


# Important Terms df

In [283]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [284]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,1025,"{MYCBP2, KIF21B, FBXL14, PITPNB, HERC4, SEC23I...",233,792,0.227317
1,1,980,{},0,980,0.000000
2,2,1200,"{TRIB1, XAF1, CD55, CLEC2D, LSP1, IRF2, CHI3L1...",594,606,0.495000
3,3,1034,"{LSM3, HYLS1, RAD21, MAD1L1, DDX18, HASPIN, MR...",775,259,0.749516
4,4,1111,"{ARHGAP4, MAPT, PEF1, PDE1B, SHANK1, WDR83, LA...",1052,59,0.946895
5,5,748,"{COL13A1, STXBP6, AGRN, GDF10, PVR, THBS2, ADA...",340,408,0.454545
6,6,666,"{KANSL3, ZNF705A, PSMD1, RNF146, EIF4E2, MORF4...",620,46,0.930931
7,7,534,"{ADRA2C, PLPPR5, ASCL4, RTN4RL2, SSTR1, KCNA5,...",303,231,0.567416
8,8,484,"{FITM1, OSBPL1A, PNPO, CYP4V2, ELOVL3, SCD, AB...",274,210,0.566116
9,9,341,{},0,341,0.000000


In [285]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [286]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")

# for category, keep only the first category and make it a string instead of a list
important_terms["Category"] = important_terms["Category"].apply(lambda x: x[0] if len(x) > 0 else "None")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1025,Ubiquitin-Protein Transferase Activity (GO:000...,60/412,4.984610e-11,catalytic activity,GO_Molecular_Function_2023,2.323827e-13,0.0,0.0,3.289508,9.569307e+01,RNF10;UBE2D4;RNF13;PPP1R11;RNF14;LTN1;UBE3B;FB...,0.145631
1046,0,1025,Antigen Processing: Ubiquitination And Proteas...,55/307,1.128290e-13,Immune System,Reactome_2022,3.151649e-16,0.0,0.0,4.212752,1.503676e+02,UBE2D4;RNF14;LTN1;UBE3B;FBXO21;HERC4;NPEPPS;RN...,0.179153
1047,0,1025,Class I MHC Mediated Antigen Processing And Pr...,55/378,4.387452e-10,Immune System,Reactome_2022,2.451091e-12,0.0,0.0,3.274265,8.753580e+01,UBE2D4;RNF14;LTN1;UBE3B;FBXO21;HERC4;NPEPPS;RN...,0.145503
1048,0,1025,Neddylation R-HSA-8951664,41/237,7.566094e-10,Metabolism of proteins,Reactome_2022,6.340303e-12,0.0,0.0,3.992134,1.029336e+02,DCUN1D3;DCUN1D4;DCUN1D1;ASB13;TULP4;DCAF5;FBXO...,0.172996
1049,0,1025,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,34/181,3.558480e-09,Vesicle-mediated transport,Reactome_2022,3.975956e-11,0.0,0.0,4.394324,1.052360e+02,ARF3;ARF4;SCOC;NBAS;TMF1;STX16;GOSR2;GOSR1;GOL...,0.187845
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
876,10,300,Sensory Perception Of Smell (GO:0007608),149/230,3.020874e-227,multicellular organismal process,GO_Biological_Process_2023,1.438511e-228,0.0,0.0,239.001799,1.253865e+05,OR8I2;OR1C1;OR2M7;OR2M5;OR2M4;OR2M3;OR2M2;OR2T...,0.647826
875,10,300,Olfactory Receptor Activity (GO:0004984),250/362,0.000000e+00,molecular transducer activity,GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,874.464286,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR2M5;OR52N1;OR2M4;OR2...,0.690608
1670,10,300,Olfactory Signaling Pathway R-HSA-381753,293/401,0.000000e+00,Sensory Perception,Reactome_2022,0.000000e+00,0.0,0.0,7593.195767,inf,OR7G1;OR8I2;OR9K2;OR2M7;OR11H4;OR2M5;OR52N1;OR...,0.730673
879,10,300,Sensory Perception Of Chemical Stimulus (GO:00...,67/110,5.372131e-95,multicellular organismal process,GO_Biological_Process_2023,1.023263e-95,0.0,0.0,131.452141,2.875155e+04,OR10J1;OR8I2;OR1C1;OR8U9;OR8U8;OR2M4;OR8U3;OR8...,0.609091


In [287]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

In [288]:
# Create folder for important terms if it doesn't exist
Path(f"../../enriched_terms/{DISEASE}").mkdir(parents=True, exist_ok=True)

In [289]:
important_terms.to_csv(f"../../enriched_terms/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [290]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [291]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [292]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [293]:
# twr3

In [294]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [295]:
# terms_with_recurrence

In [296]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [297]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [298]:
# terms_with_rec_merged

In [299]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [300]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [301]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [302]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))